# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [5]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/hieutrungdao/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


commit : 8c657d9
GPU    : Tesla T4
VRAM   : 14.6 GB


In [6]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 1.81s

7 passed · 0 warnings · 0 failures

Ready to submit.


In [3]:
# @title 3. Đánh giá đầy đủ — chạy lại NB2 và NB5
import os

COMPUTE_TIER = "T4"       # @param ["CPU", "LAPTOP", "T4", "BIGGPU"]
EVAL_LIMIT = ""           # @param ["", "4", "8", "16", "25"]
STAGES = "nb2 nb5"        # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER

# Chuỗi rỗng nghĩa là đánh giá toàn bộ dữ liệu.
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device

print(device.banner())
print("EVAL_LIMIT:", os.environ.get("EVAL_LIMIT", "FULL"))
print("STAGES:", STAGES)
print()

!python scripts/colab_run.py {STAGES}

Tesla T4 (cuda, sm_75, 14.6 GB) -> precision=fp16
  note: this GPU predates Ampere, so it has NO bfloat16. Using fp16 with gradient scaling instead. Tutorials that hardcode bf16=True fail here. 

tier=T4  mask=assistant-only  eval_limit=8

NB1 — data, chat template & loss mask
tier=T4  model=unsloth/Qwen3.5-4B  max_length=1024
250 mẫu huấn luyện
{
  "instruction": "Phân loại ticket chăm sóc khách hàng sau thành JSON với đúng 4 khóa: intent, urgency, product, sentiment. Chỉ trả về JSON, không giải thích.\n\nintent thuộc: doi_tra | van_chuyen | hoan_tien | san_pham_loi | hoi_thong_tin\nurgency thuộc: cao | trung_binh | thap\nsentiment thuộc: tieu_cuc | trung_tinh | tich_cuc\nproduct: tên sản phẩm xuất hiện trong ticket.",
  "input": "Alo sh
config.json: 100% 2.76k/2.76k [00:00<00:00, 7.02MB/s]
tokenizer_config.json: 100% 15.7k/15.7k [00:00<00:00, 31.9MB/s]

tokenizer.json: downloading bytes:   0% 0.00/20.0M [00:00<?, ?B/s]
tokenizer.json: downloading bytes: 100% 6.23M/6.23M [00:00<00:00,

In [4]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 2.49s
[  ok  ] results/template_check.json                      
[  ok  ] results/mask_proof.json                          
[  ok  ] results/token_stats.json                         
[  ok  ] results/baselines_frozen.json                    
[  ok  ] results/runs.csv                                 
[  ok  ] results/verdict.json                             
[  ok  ] results/autopsy.json                             
[  ok  ] submission/REPORT.md                             
[ FAIL ] REPORT.md filled in                     